In [41]:
#2 – Imports
from pathlib import Path
import os
import pickle

from dotenv import load_dotenv
from tqdm.auto import tqdm

from pinecone import Pinecone, ServerlessSpec

In [42]:
#3 – Load Environment
load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
assert PINECONE_API_KEY is not None

In [43]:
#4 – Paths
PROJECT_ROOT = Path("..").resolve()

ARTIFACTS = PROJECT_ROOT / "artifacts"

INPUT_FILE = ARTIFACTS / "vector_records.pkl"

In [44]:
#5 – Load Records
with open(INPUT_FILE, "rb") as f:
    vector_records = pickle.load(f)

print(len(vector_records))

5073


In [45]:
#6 – Connect to Pinecone
pc = Pinecone(
    api_key=PINECONE_API_KEY
)

In [46]:
#7 – Index Configuration
INDEX_NAME = "supportsphere-ai"

DIMENSION = len(vector_records[0]["values"])

METRIC = "cosine"

CLOUD = "aws"

REGION = "us-east-1"

In [47]:
#8 – Create Index (If Needed)
existing_indexes = [
    index.name
    for index in pc.list_indexes()
]

if INDEX_NAME not in existing_indexes:

    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric=METRIC,
        spec=ServerlessSpec(
            cloud=CLOUD,
            region=REGION
        )
    )

    print("Index Created")

else:

    print("Index Already Exists")

Index Created


In [48]:
#9 – Connect to Index
index = pc.Index(INDEX_NAME)

In [10]:
vector_records[:1]

[{'id': 'a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531',
  'values': [-0.019292394,
   0.036209315,
   -0.0054648607,
   -0.08122756,
   -0.008192824,
   0.014892825,
   -0.0033974648,
   0.018296245,
   -0.0039896686,
   -0.018080883,
   -0.017180704,
   -0.0206561,
   0.0035057694,
   -0.020340081,
   0.10035788,
   0.0034956327,
   0.018127132,
   -0.022329653,
   0.008749659,
   -7.503114e-05,
   0.022565596,
   0.03742325,
   0.024302268,
   -0.010607429,
   -0.017585538,
   -0.0053662383,
   0.02780754,
   -0.007418054,
   0.010749643,
   0.00019306071,
   0.0143946055,
   -0.016569218,
   0.001982533,
   0.023706786,
   0.006502816,
   0.012239518,
   -0.0030268263,
   -0.008487426,
   -0.0049530948,
   -0.0026423775,
   -0.003010098,
   0.009274401,
   6.318207e-05,
   0.00047386612,
   0.0015164127,
   -0.018715316,
   0.0061676307,
   -0.039318644,
   -0.009030991,
   0.010760546,
   0.0023241127,
   0.011779123,
   -0.007064618,
   -0.15658666,
   -0.02193

In [49]:
#10 – Prepare Pinecone Records
pinecone_records = []

for record in vector_records:

    metadata = record["metadata"].copy()

    metadata["text"] = record["text"]

    pinecone_records.append({

        "id": record["id"],

        "values": record["values"],

        "metadata": metadata

    })

In [50]:
#11 – Batch Upload
BATCH_SIZE = 100

for start in tqdm(range(0, len(pinecone_records), BATCH_SIZE)):

    batch = pinecone_records[start:start+BATCH_SIZE]
    index.upsert(vectors=batch)

100%|██████████| 51/51 [04:11<00:00,  4.92s/it]


In [51]:
#12 – Verify Statistics
stats = index.describe_index_stats()
stats

DescribeIndexStatsResponse(dimension=768, total_vector_count=5073, metric='cosine', namespaces=1)

In [52]:
#13 – Print Summary
print("="*60)

print("PINECONE INGESTION COMPLETE")

print("="*60)

print(f"Index : {INDEX_NAME}")

print(f"Vectors : {stats['total_vector_count']}")

print(f"Dimension : {DIMENSION}")

print(f"Metric : {METRIC}")

PINECONE INGESTION COMPLETE
Index : supportsphere-ai
Vectors : 5073
Dimension : 768
Metric : cosine


In [53]:

#14 – Verify One Vector
sample_id = vector_records[0]["id"]

response = index.fetch(
    ids=[sample_id]
)

response

FetchResponse(vectors={'a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531': Vector(id='a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531', values=[-0.0192923937, 0.0362093151, -0.0054648607, ...765 more], sparse_values=None, metadata={'chunk_id': 'a626375f498a76333041bb81874837aadeb4ef80351c4c9b04582d972e9b7531', 'chunk_number': 0, 'chunk_size': 439, 'company': 'claude', 'content_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924', 'document_hash': 'f95538b822db9ca84075a7b73f7b4ed08c2e4e087ec3c972c08f9e6565a85924', 'document_id': 'bac5cafbe3fad78362bf9f7102a348a004454d67a65cec3cf592deed2ad6d9fb', 'extension': '.md', 'file_name': '10280779-how-do-i-learn-more-about-amazon-and-anthropic-s-strategic-collaboration.md', 'indexed_at': '2026-07-07T20:51:17.786235+00:00', 'last_modified': '2026-07-02T21:53:33.772476+00:00', 'product_area': 'amazon_bedrock', 'schema_version': '1.0', 'source_path': 'claude\\amazon-bedrock\\10280779-how-do-i-lea

## Retrieval

In [54]:
#3 — Load Environment
load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")

assert PINECONE_API_KEY is not None

In [55]:
#4 — Connect to Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
INDEX_NAME = "supportsphere-ai"
index = pc.Index(INDEX_NAME)

In [57]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
#5 — Embedding Model
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    output_dimensionality=768
)

In [58]:
#6 — Query
question = "How do I reset my Claude password?"
print(question)

How do I reset my Claude password?


In [59]:
#7 — Generate Query Embedding
query_vector = embedding_model.embed_query(question)
print(len(query_vector))

768


In [60]:
len(query_vector)

768

In [61]:
stats = index.describe_index_stats()

print(stats)

DescribeIndexStatsResponse(dimension=768, total_vector_count=5073, metric='cosine', namespaces=1)


In [62]:
#8 — Basic Semantic Search
results = index.query(vector=query_vector, top_k=5,include_metadata=True)

In [63]:
#9 — Inspect Response
results

QueryResponse(matches=[ScoredVector(id='ae72c78571ed70e2ee37f59cd25b31d9a7d8ab10bb4706bfd84b5d79583d19b5', score=0.78196764, values=[], metadata={'chunk_id': 'ae72c78571ed70e2ee37f59cd25b31d9a7d8ab10bb4706bfd84b5d79583d19b5', 'chunk_number': 6, 'chunk_size': 954, 'company': 'claude', 'content_hash': '25a8b5f6348bc590d31c5e238a8c7162196a8ebc1ce3df0cf4e733e8407f1266', 'document_hash': 'cfcee173dd1a5c978f034bae7aa6ad5d3ac2a2c384d95ca0de3a83beb4bbcae4', 'document_id': 'bf561ee19e8c0f0aa01e79ad585aabe8d048317fbfd02e9237464e8895a929c6', 'extension': '.md', 'file_name': '13189465-logging-in-to-your-claude-account.md', 'indexed_at': '2026-07-07T20:51:17.800845+00:00', 'last_modified': '2026-07-02T21:53:33.823191+00:00', 'product_area': 'claude', 'schema_version': '1.0', 'source_path': 'claude\\claude\\account-management\\13189465-logging-in-to-your-claude-account.md', 'source_type': 'markdown', 'text': 'How can I set a password for my Claude account?\nIt’s not possible to create a dedicated pa

In [64]:
#10 — Pretty Print Results
for rank, match in enumerate(results["matches"], start=1):

    print("="*100)

    print(f"Rank : {rank}")

    print(f"Score : {match['score']:.4f}")

    print()

    print("Company :", match["metadata"]["company"])

    print("Product :", match["metadata"]["product_area"])

    print("Title :", match["metadata"]["title"])

    print()

    print(match["metadata"]["text"][:600])

Rank : 1
Score : 0.7820

Company : claude
Product : claude
Title : "Logging in to your Claude account"

How can I set a password for my Claude account?
It’s not possible to create a dedicated password for your Claude account at this time.
Can I use my Claude account across multiple devices?
To use your Claude account across multiple devices, enter the same email address you use to log in on your usual device. For example, if you signed up for your Claude account on a web browser, you can access the same account on the Claude mobile apps by entering the same email address when you log in.
I have multiple accounts – how can I switch between them?
If you have both an individual Claude account and a
Rank : 2
Score : 0.7722

Company : claude
Product : claude
Title : "Logging in to your Claude account"

**Note: **Business, government, or .edu email users may need to contact their IT department to adjust email security settings.
I received the email but I'm still having trouble logging in.
If

In [66]:
#11 — Company Filter
results = index.query(

    vector=query_vector,

    top_k=5,

    include_metadata=True,

    filter={

        "company":"claude"

    }

)

In [67]:
for match in results["matches"]:

    print(match["metadata"]["company"])

    print(match["metadata"]["title"])

    print()

claude
"Logging in to your Claude account"

claude
"Logging in to your Claude account"

claude
"Logging in to your Claude account"

claude
"Logging in to your Console account"

claude
"Logging in to your Claude account"



In [ ]:
#12 — Visa Filter
question = "My Visa card was declined."

query_vector = embedding_model.embed_query(question)

results = index.query(vector=query_vector,top_k=5,include_metadata=True,
                      filter={"company":"visa"})

In [69]:
for match in results["matches"]:

    print(match["metadata"]["title"])

"Visa Consumer Support"
"Visa Consumer Support"
"Visa Consumer Support"
"Visa Consumer Support"
"Visa Credit Card Rules & Regulations"


In [70]:
#13 — Product Area Filter
question = "How do I access Claude in Amazon Bedrock?"

query_vector = embedding_model.embed_query(question)

results = index.query(

    vector=query_vector,

    top_k=5,

    include_metadata=True,

    filter={

        "company":"claude",

        "product_area":"amazon_bedrock"

    }

)

In [71]:
for match in results["matches"]:

    print("="*80)

    print(match["metadata"]["title"])

    print()

    print(match["metadata"]["text"][:500])

"How do I get access to Claude in Amazon Bedrock?"

How do I get access to Claude in Amazon Bedrock?
_Last updated: 2026-03-16T21:16:50Z_
Get started with Claude in Amazon Bedrock by visiting the Amazon Bedrock console. For a step-by-step walkthrough on how to request Claude model access in the Amazon Bedrock console, view this blog.
Related Articles
What is Amazon Bedrock?
I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?
Where do I find Claude in Amazon Bedrock documentation?
What AWS Regions are Claude models ava
"I use Claude in Amazon Bedrock. Who do I contact for customer support inquiries?"

Related Articles
What is Amazon Bedrock?
What AWS Regions are Claude models available in Amazon Bedrock?
Claude Code FAQ
Public Sector FAQs
Use Claude for Excel, PowerPoint, and Word with third-party platforms
"What is Amazon Bedrock?"

You can learn more about Anthropic’s Claude models in Amazon Bedrock here.
Related Articles
How do I get access to Claude in A

In [72]:
#14 — Compare Retrieval
results_all = index.query(

    vector=query_vector,

    top_k=5,

    include_metadata=True
)

In [73]:
results_filtered = index.query(

    vector=query_vector,

    top_k=5,

    include_metadata=True,

    filter={

        "company":"claude"
    }
)

In [74]:
print("Without Filter")

for match in results_all["matches"]:

    print(match["metadata"]["company"])

Without Filter
claude
claude
claude
claude
claude


In [75]:
print()

print("With Filter")

for match in results_filtered["matches"]:

    print(match["metadata"]["company"])


With Filter
claude
claude
claude
claude
claude


In [76]:
#15 — Helper Function
def retrieve_documents(
    question: str,
    company: str | None = None,
    product_area: str | None = None,
    top_k: int = 5,
):

    query_vector = embedding_model.embed_query(question)

    filter_dict = {}

    if company:

        filter_dict["company"] = company.lower()

    if product_area:

        filter_dict["product_area"] = product_area

    results = index.query(

        vector=query_vector,

        top_k=top_k,

        include_metadata=True,

        filter=filter_dict if filter_dict else None

    )

    return results["matches"]

In [77]:
#16 — Test Helper
matches = retrieve_documents(

    question="How do I change my password?",

    company="claude",

    top_k=3
)

In [81]:
for match in matches:

    print(match["metadata"]["title"])

    print(match["score"])
    print(match["metadata"]["company"])

    print()

"Logging in to your Console account"
0.639748335
claude

"Verifying your phone number"
0.626375318
claude

"Logging in to your Claude account"
0.625305831
claude



In [82]:
#17 — Build Contex
context = "\n\n".join(

    match["metadata"]["text"]

    for match in matches

)
print(context[:2000])

How can I set a password for my Console account?
It's not possible to create a dedicated password for your Console account at this time.
Does my organization use Single Sign-On (SSO)?
If you're a member of an organization that has configured single sign-on for the Console, you'll be redirected to your SSO provider when you attempt to log in. For information about setting up SSO for your organization, see Set up single sign-on (SSO).
I have multiple Console accounts – how can I switch between them?
If you're a member of multiple Console organizations tied to the same email address, follow these steps to switch to your other account:
Click your initials in the bottom left corner to open a menu showing all the accounts tied to your logged-in email address.
Select the desired account.
Look for the check mark indicating the active account.
I don't want to use "Continue with Google" to log in anymore; how can I revoke access to my Google account?

Alternatively, you can enter a different pho

In [83]:
#18 — Notebook Summary
print("="*60)

print("PINECONE RETRIEVAL COMPLETE")

print("="*60)

print("Retriever Ready")

print()

print("Semantic Search ✓")

print("Metadata Filtering ✓")

print("Top-K Retrieval ✓")

print("Context Builder ✓")

PINECONE RETRIEVAL COMPLETE
Retriever Ready

Semantic Search ✓
Metadata Filtering ✓
Top-K Retrieval ✓
Context Builder ✓


In [31]:
"""
matches = retrieve_documents(
    state["question"],
    state.get("company"),
    state.get("product_area"),
)

context = "\n\n".join(
    match["metadata"]["text"]
    for match in matches
)

return {
    "documents": matches,
    "context": context,
}
"""

'\nmatches = retrieve_documents(\n    state["question"],\n    state.get("company"),\n    state.get("product_area"),\n)\n\ncontext = "\n\n".join(\n    match["metadata"]["text"]\n    for match in matches\n)\n\nreturn {\n    "documents": matches,\n    "context": context,\n}\n'